# Round-trips vs. one-way

Follow-up to `01_first_look.ipynb`, which showed the top 20 OD pairs in Jan 2025 were *all* round-trips. Two hypotheses:

1. Short-duration "aborts" — grabbed a bike, changed mind, re-docked immediately.
2. Genuine leisure loops — rode around and came back.

We'll split the question by duration, user type, and then strip round-trips to find the real commute corridors.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import duckdb
import pandas as pd

DATA_RAW = Path.cwd().parent / 'data' / 'raw'
GLOB = str(DATA_RAW / 'ridership' / '2025' / '*.csv')

con = duckdb.connect()
con.execute(f"CREATE VIEW trips AS SELECT * FROM read_csv_auto('{GLOB}', header=True, ignore_errors=true, union_by_name=true)")
con.execute('SELECT COUNT(*) AS n, MIN(Start_Time) AS lo, MAX(Start_Time) AS hi FROM trips').fetchdf()

,n,lo,hi
0,7812520,2025-01-01 00:01:51,2025-12-31 23:59:49


## 1. Round-trip share overall

In [2]:
con.execute('''
    SELECT
      COUNT(*) AS total_trips,
      SUM(CASE WHEN Start_Station_Id = End_Station_Id THEN 1 ELSE 0 END) AS round_trips,
      ROUND(100.0 * SUM(CASE WHEN Start_Station_Id = End_Station_Id THEN 1 ELSE 0 END) / COUNT(*), 2) AS round_trip_pct
    FROM trips
    WHERE Start_Station_Id IS NOT NULL AND End_Station_Id IS NOT NULL
''').fetchdf()

,total_trips,round_trips,round_trip_pct
0,7805759,247238.0,3.17


## 2. Round-trip duration distribution

Trip_Duration is in seconds. Does the round-trip share skew heavily to very short durations (aborts)?

In [3]:
con.execute('''
    WITH b AS (
      SELECT
        CASE
          WHEN Trip_Duration < 60   THEN '0_under_1min'
          WHEN Trip_Duration < 120  THEN '1_1_to_2min'
          WHEN Trip_Duration < 300  THEN '2_2_to_5min'
          WHEN Trip_Duration < 600  THEN '3_5_to_10min'
          WHEN Trip_Duration < 1800 THEN '4_10_to_30min'
          WHEN Trip_Duration < 3600 THEN '5_30_to_60min'
          ELSE '6_over_1hr'
        END AS bucket,
        (Start_Station_Id = End_Station_Id) AS is_round
      FROM trips
      WHERE Start_Station_Id IS NOT NULL AND End_Station_Id IS NOT NULL
    )
    SELECT
      bucket,
      COUNT(*) AS trips,
      SUM(CASE WHEN is_round THEN 1 ELSE 0 END) AS round_trips,
      ROUND(100.0 * SUM(CASE WHEN is_round THEN 1 ELSE 0 END) / COUNT(*), 1) AS round_pct
    FROM b
    GROUP BY 1
    ORDER BY 1
''').fetchdf()

,bucket,trips,round_trips,round_pct
0,0_under_1min,16905,2185.0,12.9
1,1_1_to_2min,78836,19084.0,24.2
2,2_2_to_5min,891543,17914.0,2.0
3,3_5_to_10min,2353795,21943.0,0.9
4,4_10_to_30min,3725232,61332.0,1.6
5,5_30_to_60min,528262,59432.0,11.3
6,6_over_1hr,211186,65348.0,30.9


## 3. Member vs. casual split

In [4]:
con.execute('''
    SELECT
      User_Type,
      COUNT(*) AS total_trips,
      SUM(CASE WHEN Start_Station_Id = End_Station_Id THEN 1 ELSE 0 END) AS round_trips,
      ROUND(100.0 * SUM(CASE WHEN Start_Station_Id = End_Station_Id THEN 1 ELSE 0 END) / COUNT(*), 2) AS round_pct
    FROM trips
    WHERE Start_Station_Id IS NOT NULL AND End_Station_Id IS NOT NULL
    GROUP BY 1
    ORDER BY total_trips DESC
''').fetchdf()

,User_Type,total_trips,round_trips,round_pct
0,Member,5667704,88397.0,1.56
1,Casual,2138055,158841.0,7.43


## 4. Abort-looking round-trips

Round-trips under 2 minutes almost certainly aren't real rides. What share of all round-trips is that?

In [5]:
con.execute('''
    SELECT
      COUNT(*) AS round_trips,
      SUM(CASE WHEN Trip_Duration < 120 THEN 1 ELSE 0 END) AS abort_looking,
      ROUND(100.0 * SUM(CASE WHEN Trip_Duration < 120 THEN 1 ELSE 0 END) / COUNT(*), 1) AS abort_pct,
      ROUND(AVG(Trip_Duration) / 60.0, 1) AS mean_minutes,
      ROUND(median(Trip_Duration) / 60.0, 1) AS median_minutes
    FROM trips
    WHERE Start_Station_Id = End_Station_Id
      AND Start_Station_Id IS NOT NULL
''').fetchdf()

,round_trips,abort_looking,abort_pct,mean_minutes,median_minutes
0,247238,21269.0,8.6,43.8,30.4


## 5. Top one-way OD pairs (round-trips excluded)

This is the real commute/directional-flow signal.

In [6]:
con.execute('''
    SELECT
      Start_Station_Name AS origin,
      End_Station_Name AS dest,
      COUNT(*) AS trips
    FROM trips
    WHERE Start_Station_Id <> End_Station_Id
      AND Start_Station_Name IS NOT NULL AND End_Station_Name IS NOT NULL
    GROUP BY 1, 2
    ORDER BY trips DESC
    LIMIT 20
''').fetchdf()

,origin,dest,trips
0,York St / Queens Quay W,York St / Queens Quay W,48501
1,Centre Island Ferry Dock,Centre Island Ferry Dock,42958
2,Widmer St / Adelaide St W - SMART,Widmer St / Adelaide St W - SMART,37610
3,College St / Major St,College St / Major St,35283
4,King St W / Portland St - SMART,King St W / Portland St - SMART,34961
5,Bay St / College St (East Side),Bay St / College St (East Side),34783
6,Frederick St / King St E,Frederick St / King St E,34066
7,Bay St / Queens Quay W (Ferry Terminal),Bay St / Queens Quay W (Ferry Terminal),33712
8,Queen St W / John St,Queen St W / John St,33300
9,Bay St / Wellesley St W,Bay St / Wellesley St W,32414


## 6. Top one-way OD pairs, collapsing A↔B into a single bidirectional pair

For a commute corridor we don't care about direction — sum both directions and show imbalance.

In [7]:
con.execute('''
    WITH pairs AS (
      SELECT
        LEAST(Start_Station_Name, End_Station_Name) AS a,
        GREATEST(Start_Station_Name, End_Station_Name) AS b,
        CASE WHEN Start_Station_Name < End_Station_Name THEN 1 ELSE 0 END AS a_to_b,
        CASE WHEN Start_Station_Name > End_Station_Name THEN 1 ELSE 0 END AS b_to_a
      FROM trips
      WHERE Start_Station_Id <> End_Station_Id
        AND Start_Station_Name IS NOT NULL AND End_Station_Name IS NOT NULL
    )
    SELECT a, b,
           SUM(a_to_b) AS a_to_b,
           SUM(b_to_a) AS b_to_a,
           SUM(a_to_b + b_to_a) AS total,
           ROUND(100.0 * ABS(SUM(a_to_b) - SUM(b_to_a)) / NULLIF(SUM(a_to_b + b_to_a), 0), 1) AS imbalance_pct
    FROM pairs
    GROUP BY 1, 2
    ORDER BY total DESC
    LIMIT 20
''').fetchdf()

,a,b,a_to_b,b_to_a,total,imbalance_pct
0,Grand Avenue Park,Windsor St / Newcastle St,237.0,204.0,441.0,7.5
1,Frederick St / King St E,King St E / Victoria St,159.0,133.0,292.0,8.9
2,Bay St / Wellesley St W,Huron St / Harbord St,167.0,99.0,266.0,25.6
3,King St W / Bay St (West Side),King St W / Portland St - SMART,77.0,187.0,264.0,41.7
4,Bay St / College St (East Side),College St / Huron St,123.0,122.0,245.0,0.4
5,King St W / Bay St (West Side),King St W / Charlotte St,62.0,175.0,237.0,47.7
6,Front St W / Blue Jays Way,Union Station,158.0,77.0,235.0,34.5
7,Frederick St / King St E,Front St W / Bay St (South Side),134.0,101.0,235.0,14.0
8,College St / Major St,Robert St / Bloor St W - SMART,132.0,95.0,227.0,16.3
9,Bathurst St/Queens Quay(Billy Bishop Airport),York St / Queens Quay W,114.0,107.0,221.0,3.2


## 7. Member vs. casual — does the round-trip pattern change the OD head?

In [8]:
con.execute('''
    SELECT
      User_Type,
      Start_Station_Name AS origin,
      End_Station_Name AS dest,
      COUNT(*) AS trips
    FROM trips
    WHERE Start_Station_Id <> End_Station_Id
      AND Start_Station_Name IS NOT NULL AND End_Station_Name IS NOT NULL
    GROUP BY 1, 2, 3
    QUALIFY ROW_NUMBER() OVER (PARTITION BY User_Type ORDER BY COUNT(*) DESC) <= 10
    ORDER BY User_Type, trips DESC
''').fetchdf()

,User_Type,origin,dest,trips
0,Casual,Centre Island Ferry Dock,Centre Island Ferry Dock,39837
1,Casual,Ward's Island Ferry Dock,Ward's Island Ferry Dock,23479
2,Casual,Centre Island,Centre Island,20837
3,Casual,Hanlan's Point Ferry Dock,Hanlan's Point Ferry Dock,19547
4,Casual,York St / Queens Quay W,York St / Queens Quay W,18514
5,Casual,Hanlan's Point Beach,Hanlan's Point Beach,15258
6,Casual,Bay St / Queens Quay W (Ferry Terminal),Bay St / Queens Quay W (Ferry Terminal),14730
7,Casual,Queens Quay / Yonge St,Queens Quay / Yonge St,12257
8,Casual,Bremner Blvd / Rees St,Bremner Blvd / Rees St,12248
9,Casual,Gibraltar Point Beach,Gibraltar Point Beach,12117
